<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/DS_PROJECT/ds_proj_meth/CRISP_DM_Project_Sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проект: Прогнозирование стоимости жилья с помощью нейронной сети  
**Автор:** Кондратьев Степан  
**Дата:** апрель 2025  
**Цель:** Реализация и обучение простейшей нейронной сети для предсказания стоимости квартир на данных Boston House Prices или California Housing с использованием PyTorch.

## 1. Понимание бизнеса (Business Understanding)  

**Цель проекта:**  
Прогнозирование стоимости жилья на основе характеристик недвижимости для помощи риелторам, покупателям и застройщикам в принятии обоснованных решений.  

**Постановка задачи:**  
Регрессионная задача предсказания цены квартиры/дома на основе таких параметров, как площадь, количество комнат, местоположение и других значимых признаков из датасетов Boston House Prices или California Housing.  

**Критерии успеха:**  
- **Метрики:**  
  - MAE (Mean Absolute Error)  
  - MSE (Mean Squared Error)  
  - R² (Коэффициент детерминации)  
- **Бизнес-результат:**  
  - Снижение ошибки оценки стоимости недвижимости на 15-20% по сравнению с ручными методами.  
  - Ускорение процесса оценки для риелторов и покупателей.  


## 2. Понимание данных (Data Understanding)

### Импорт библиотек

In [1]:
# Импорт библиотек
import polars as pl  # Для загрузки и обработки данных (альтернатива pandas)
import torch  # Основной фреймворк для работы с нейронными сетями
from torch import nn  # Для создания нейросетевых слоев
from sklearn.model_selection import train_test_split  # Для разделения данных на train/test
from sklearn.preprocessing import StandardScaler  # Для стандартизации данных
from sklearn.datasets import fetch_california_housing  # Для загрузки датасета California Housing

### Загрузка и первичный анализ данных

In [2]:
# Настройка отображения
pl.Config.set_tbl_cols(-1)  # Показать все столбцы
pl.Config.set_tbl_width_chars(120)  # Ширина таблицы
pl.Config.set_tbl_rows(5)  # Ограничить количество выводимых строк


polars.config.Config

In [3]:
# Загрузка данных
california = fetch_california_housing(as_frame=True)
df = pl.DataFrame(california.data)
df = df.with_columns(pl.Series("MedHouseVal", california.target))  # Добавляем целевую переменную

In [4]:
# Вывод данных для первичного анализа
print("Данные обучающей выборки:")
print(df)

Данные обучающей выборки:
shape: (20_640, 9)
┌────────┬──────────┬──────────┬───────────┬────────────┬──────────┬──────────┬───────────┬─────────────┐
│ MedInc ┆ HouseAge ┆ AveRooms ┆ AveBedrms ┆ Population ┆ AveOccup ┆ Latitude ┆ Longitude ┆ MedHouseVal │
│ ---    ┆ ---      ┆ ---      ┆ ---       ┆ ---        ┆ ---      ┆ ---      ┆ ---       ┆ ---         │
│ f64    ┆ f64      ┆ f64      ┆ f64       ┆ f64        ┆ f64      ┆ f64      ┆ f64       ┆ f64         │
╞════════╪══════════╪══════════╪═══════════╪════════════╪══════════╪══════════╪═══════════╪═════════════╡
│ 8.3252 ┆ 41.0     ┆ 6.984127 ┆ 1.02381   ┆ 322.0      ┆ 2.555556 ┆ 37.88    ┆ -122.23   ┆ 4.526       │
│ 8.3014 ┆ 21.0     ┆ 6.238137 ┆ 0.97188   ┆ 2401.0     ┆ 2.109842 ┆ 37.86    ┆ -122.22   ┆ 3.585       │
│ 7.2574 ┆ 52.0     ┆ 8.288136 ┆ 1.073446  ┆ 496.0      ┆ 2.80226  ┆ 37.85    ┆ -122.24   ┆ 3.521       │
│ …      ┆ …        ┆ …        ┆ …         ┆ …          ┆ …        ┆ …        ┆ …         ┆ …           │
│

```
Данные обучающей выборки:
shape: (20_640, 9)
┌────────┬──────────┬──────────┬───────────┬────────────┬──────────┬──────────┬───────────┬─────────────┐
│ MedInc ┆ HouseAge ┆ AveRooms ┆ AveBedrms ┆ Population ┆ AveOccup ┆ Latitude ┆ Longitude ┆ MedHouseVal │
│ ---    ┆ ---      ┆ ---      ┆ ---       ┆ ---        ┆ ---      ┆ ---      ┆ ---       ┆ ---         │
│ f64    ┆ f64      ┆ f64      ┆ f64       ┆ f64        ┆ f64      ┆ f64      ┆ f64       ┆ f64         │
╞════════╪══════════╪══════════╪═══════════╪════════════╪══════════╪══════════╪═══════════╪═════════════╡
│ 8.3252 ┆ 41.0     ┆ 6.984127 ┆ 1.02381   ┆ 322.0      ┆ 2.555556 ┆ 37.88    ┆ -122.23   ┆ 4.526       │
│ 8.3014 ┆ 21.0     ┆ 6.238137 ┆ 0.97188   ┆ 2401.0     ┆ 2.109842 ┆ 37.86    ┆ -122.22   ┆ 3.585       │
│ 7.2574 ┆ 52.0     ┆ 8.288136 ┆ 1.073446  ┆ 496.0      ┆ 2.80226  ┆ 37.85    ┆ -122.24   ┆ 3.521       │
│ …      ┆ …        ┆ …        ┆ …         ┆ …          ┆ …        ┆ …        ┆ …         ┆ …           │
│ 1.8672 ┆ 18.0     ┆ 5.329513 ┆ 1.17192   ┆ 741.0      ┆ 2.123209 ┆ 39.43    ┆ -121.32   ┆ 0.847       │
│ 2.3886 ┆ 16.0     ┆ 5.254717 ┆ 1.162264  ┆ 1387.0     ┆ 2.616981 ┆ 39.37    ┆ -121.24   ┆ 0.894   
```

### Описание переменных

1. **MedInc**: Средний доход населения в районе (в десятках тысяч долларов)
2. **HouseAge**: Средний возраст домов в районе (в годах)
3. **AveRooms**: Среднее количество комнат на одно жилище
4. **AveBedrms**: Среднее количество спален на одно жилище
5. **Population**: Численность населения в районе
6. **AveOccup**: Среднее количество жителей на одно жилище
7. **Latitude**: Географическая широта района
8. **Longitude**: Географическая долгота района
9. **MedHouseVal**: Медианная стоимость дома в районе (в сотнях тысяч долларов) - целевая переменная

### Анализ переменных по типу

#### Анализ бинарных переменных

In [5]:
# Анализ бинарных переменных
print("Анализ бинарных переменных:")

binary_count = 0  # Счетчик бинарных переменных
for col in df.columns:
    unique_vals = df[col].unique().drop_nulls()  # Уникальные значения без пропусков
    if len(unique_vals) == 2:  # Проверка на бинарность
        dtype = str(df[col].dtype)  # Получаем тип данных колонки
        sorted_vals = sorted(unique_vals, key=lambda x: len(str(x)))  # Сортировка по длине строки
        print(f"Бинарная переменная найдена: {col} ({dtype}): {sorted_vals}")
        binary_count += 1

if binary_count == 0:  # Если не найдено бинарных переменных
    print("Бинарные переменные не обнаружены")

print(f"\nИтого найдено бинарных переменных: {binary_count}")

Анализ бинарных переменных:
Бинарные переменные не обнаружены

Итого найдено бинарных переменных: 0


```
Анализ бинарных переменных:
Бинарные переменные не обнаружены

Итого найдено бинарных переменных: 0
```